# 🔧 LMVD Label Fix

**Problem**: All 1,823 LMVD samples labeled as Normal (0 depressed)

**This notebook**:
1. Examines LMVD raw data to find correct labels
2. Updates existing H5 files with correct labels
3. Regenerates labels CSV

In [ ]:
# Step 1: Mount Drive
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted")

In [ ]:
# Step 2: Examine LMVD raw data structure
from pathlib import Path
import pandas as pd
import numpy as np

LMVD_RAW = Path("/content/drive/MyDrive/DAIC-WOZ_Datasets/LMVD_Raw")
LMVD_OUTPUT = Path("/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output/LMVD")

print("📂 Examining LMVD_Raw...")

# Find all file types
files = list(LMVD_RAW.rglob("*"))
exts = {}
for f in files:
    if f.is_file():
        ext = f.suffix.lower()
        exts[ext] = exts.get(ext, 0) + 1

print(f"\nFile types found:")
for ext, count in sorted(exts.items(), key=lambda x: -x[1])[:10]:
    print(f"  {ext}: {count}")

# Check for CSV files with labels
csv_files = list(LMVD_RAW.rglob("*.csv"))
print(f"\n📋 CSV files found: {len(csv_files)}")
for csv_file in csv_files[:5]:
    print(f"  - {csv_file.name}")

In [ ]:
# Step 3: Analyze CSV files for label columns
print("📊 Analyzing CSV files...\n")

for csv_file in csv_files:
    try:
        df = pd.read_csv(csv_file, nrows=10)
        print(f"📄 {csv_file.name}")
        print(f"   Shape: {df.shape}")
        print(f"   Columns: {list(df.columns)[:8]}")
        
        # Look for label-like columns
        for col in df.columns:
            col_lower = col.lower()
            if any(x in col_lower for x in ['label', 'depression', 'class', 'target', 'phq', 'diagnosis']):
                print(f"   🎯 Potential label column: {col}")
                if df[col].dtype in ['int64', 'float64']:
                    print(f"      Values: {df[col].unique()[:10]}")
        print()
    except Exception as e:
        print(f"❌ {csv_file.name}: {e}\n")

In [ ]:
# Step 4: Check if filenames contain label info
print("📂 Checking filename patterns...\n")

npy_files = list(LMVD_RAW.glob("*.npy"))
print(f"Total .npy files: {len(npy_files)}")

# Sample filenames
for f in npy_files[:20]:
    print(f"  {f.name}")

# Check for patterns suggesting depression
dep_patterns = ['dep', 'patient', 'pos', 'mdd', 'case', 'depressed']
normal_patterns = ['normal', 'control', 'neg', 'healthy', 'hc']

dep_count = sum(1 for f in npy_files if any(p in f.stem.lower() for p in dep_patterns))
normal_count = sum(1 for f in npy_files if any(p in f.stem.lower() for p in normal_patterns))

print(f"\nFilename pattern matches:")
print(f"  Depressed patterns: {dep_count}")
print(f"  Normal patterns: {normal_count}")
print(f"  No match: {len(npy_files) - dep_count - normal_count}")

In [ ]:
# Step 5: Build label dictionary from available info
# This cell tries multiple strategies to get labels

labels_dict = {}

# Strategy 1: Try to read labels from CSV
print("🔍 Strategy 1: Reading labels from CSV files...")
for csv_file in csv_files:
    try:
        df = pd.read_csv(csv_file)
        
        # Find ID column
        id_col = None
        for c in df.columns:
            if any(x in c.lower() for x in ['id', 'name', 'file', 'video', 'subject']):
                id_col = c
                break
        
        # Find label column
        label_col = None
        for c in df.columns:
            if any(x in c.lower() for x in ['label', 'depression', 'class', 'target', 'diagnosis']):
                label_col = c
                break
        
        if id_col and label_col:
            for _, row in df.iterrows():
                file_id = str(row[id_col]).replace('.npy', '').replace('.mp4', '')
                label = int(row[label_col])
                labels_dict[file_id] = label
            print(f"   ✅ Loaded {len(df)} labels from {csv_file.name}")
            print(f"      ID column: {id_col}")
            print(f"      Label column: {label_col}")
    except Exception as e:
        print(f"   ❌ {csv_file.name}: {e}")

# Strategy 2: Use filename patterns as fallback
if not labels_dict:
    print("\n🔍 Strategy 2: Using filename patterns...")
    for npy_file in npy_files:
        name = npy_file.stem.lower()
        if any(p in name for p in ['dep', 'patient', 'pos', 'mdd', 'case']):
            labels_dict[npy_file.stem] = 1  # Depressed
        else:
            labels_dict[npy_file.stem] = 0  # Normal
    print(f"   Assigned {sum(labels_dict.values())} as depressed, {len(labels_dict) - sum(labels_dict.values())} as normal")

print(f"\n📋 Total labels: {len(labels_dict)}")
print(f"   Depressed: {sum(labels_dict.values())}")
print(f"   Normal: {len(labels_dict) - sum(labels_dict.values())}")

In [ ]:
# Step 6: Update existing H5 files with correct labels
import h5py
from tqdm import tqdm

h5_files = list(LMVD_OUTPUT.glob("*.h5"))
print(f"📦 Total H5 files to update: {len(h5_files)}")

updated = 0
dep_count = 0
normal_count = 0

for h5_file in tqdm(h5_files, desc="Updating labels"):
    try:
        # Get the original filename (lmvd_XXX -> XXX)
        pid = h5_file.stem
        original_id = pid.replace('lmvd_', '')
        
        # Find label
        label = labels_dict.get(original_id, labels_dict.get(pid, -1))
        
        # If not found, use filename pattern
        if label == -1:
            name = original_id.lower()
            if any(p in name for p in ['dep', 'patient', 'pos', 'mdd', 'case']):
                label = 1
            else:
                label = 0
        
        # Update H5 file
        with h5py.File(h5_file, 'r+') as f:
            if 'label' in f:
                del f['label']
            f.create_dataset('label', data=label)
            
            # Also update phq8_score
            if 'phq8_score' in f:
                del f['phq8_score']
            f.create_dataset('phq8_score', data=15.0 if label else 3.0)
        
        if label == 1:
            dep_count += 1
        else:
            normal_count += 1
        updated += 1
        
    except Exception as e:
        print(f"\n❌ {h5_file.name}: {e}")

print(f"\n✅ Updated {updated} H5 files")
print(f"   Depressed: {dep_count}")
print(f"   Normal: {normal_count}")

In [ ]:
# Step 7: Regenerate labels CSV
labels_data = []

for h5_file in LMVD_OUTPUT.glob("*.h5"):
    try:
        with h5py.File(h5_file, 'r') as f:
            label = int(f['label'][()])
            phq8 = float(f['phq8_score'][()]) if 'phq8_score' in f else (15 if label else 3)
            labels_data.append({
                'Participant_ID': h5_file.stem,
                'PHQ8_Score': phq8,
                'PHQ8_Binary': label,
                'Source': 'LMVD'
            })
    except: pass

if labels_data:
    df = pd.DataFrame(labels_data)
    df.to_csv("/content/drive/MyDrive/DAIC-WOZ_Datasets/lmvd_labels.csv", index=False)
    
    print("="*50)
    print("🏆 LMVD LABELS FIXED")
    print("="*50)
    print(f"📦 Total: {len(df)}")
    print(f"   Depressed: {len(df[df['PHQ8_Binary']==1])}")
    print(f"   Normal: {len(df[df['PHQ8_Binary']==0])}")
    print(f"📋 Saved: /content/drive/MyDrive/DAIC-WOZ_Datasets/lmvd_labels.csv")
else:
    print("❌ No labels found")